In [90]:
import os
import yaml
import polars as pl
import altair as alt

In [91]:
pl.Config(tbl_rows=100)
pl.Config(fmt_str_lengths=150)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [92]:
df = pl.read_parquet(
    "data/segments.parquet"
).sort(
    ["id", "date"]
).with_columns(
    days_diff = pl.col("date").diff().dt.total_days().over("id"),
    effort_diff = pl.col("effort_count").diff().over("id")
).with_columns(
    avg_daily_efforts = pl.col("effort_diff") / pl.col("days_diff")
).with_columns(
    pl.when(pl.col("city").str.contains("Praha") | pl.col("city").str.contains("Prague")).then(pl.lit('Praha')).when(pl.col('city').str.contains('Brno')).then(pl.lit('Brno')).otherwise(pl.col('city')).alias('city'),
    pl.when(pl.col("effort_diff") < 0).then(pl.lit(0)).otherwise(pl.col("effort_diff")).alias("effort_diff")
).with_columns(
    pl.col('date').dt.ordinal_day().alias('day'),
    pl.col('date').dt.week().alias('week'),
    pl.col('date').dt.year().alias('year')
)

In [159]:
df.columns

['id',
 'name',
 'activity_type',
 'distance',
 'average_grade',
 'elevation_high',
 'elevation_low',
 'total_elevation_gain',
 'start_latlng',
 'end_latlng',
 'country',
 'state',
 'city',
 'effort_count',
 'athlete_count',
 'date',
 'start_to_finish_distance',
 'days_diff',
 'effort_diff',
 'avg_daily_efforts',
 'day',
 'week',
 'year']

In [93]:
print(df.select(pl.col("date").min()).item())
print(df.select(pl.col("date").max()).item())

2024-12-24 23:05:01+01:00
2026-03-08 23:39:05+01:00


In [185]:
with open("data/behani_top_20.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    celorepublika = yaml.safe_load(file)

with open("data/kolo_top_20.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    celorepublika_kolo = yaml.safe_load(file)
    
with open("data/kolo_top_30.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    celorepublika_kolo_30 = yaml.safe_load(file)

with open("data/ovaly.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    ovaly = yaml.safe_load(file)

In [95]:
orez_extremu = df.filter(
    pl.col('name').is_in(celorepublika)
).select(
    pl.col('effort_diff')
).quantile(
    0.99
).item(
)

do_grafu_celorepublika = df.filter(
    pl.col('name').is_in(celorepublika)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu).then(orez_extremu).otherwise(pl.col('effort_diff')).alias('effort_diff')
).group_by(
    ['year','week']
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['year','week']
).filter(
    pl.col('year') >= 2025
).with_columns(
    (pl.col('effort_diff') / pl.col('effort_diff').shift(1)).alias('trend')
)

do_grafu_celorepublika.filter(pl.col("trend") > 1.2)

year,week,effort_diff,trend
i32,i8,f64,f64
2025,44,1460.0,1.418853
2025,47,1615.0,1.78453
2026,2,1024.0,1.410468
2026,6,1348.0,1.454153
2026,8,1188.0,1.346939
2026,9,1589.0,1.337542


In [96]:
orez_extremu_kolo = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).select(
    pl.col('effort_diff')
).quantile(
    0.99
).item(
)

do_grafu_celorepublika_kolo = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu_kolo).then(orez_extremu_kolo).otherwise(pl.col('effort_diff')).alias('effort_diff')
).group_by(
    ['year','week']
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['year','week']
).filter(
    pl.col('year') >= 2025
).with_columns(
    (pl.col('effort_diff') / pl.col('effort_diff').shift(1)).alias('trend')
)

do_grafu_celorepublika_kolo.filter(pl.col("trend") > 1.2)

do_grafu_celorepublika_kolo_2w = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu_kolo).then(orez_extremu_kolo).otherwise(pl.col('effort_diff')).alias('effort_diff')
).sort(
    by="date"
).group_by_dynamic(
    index_column="date", every="2w"
).agg(
    pl.col('effort_diff').sum()
).sort(
    by="date"
).filter(
    pl.col('date').dt.year() >= 2025
).with_columns(
    (pl.col('effort_diff') / pl.col('effort_diff').shift(1)).alias('trend')
).with_columns(
    pl.col("date").dt.year().alias("year")
)

do_grafu_celorepublika_kolo.filter(pl.col("trend") > 1.2)

year,week,effort_diff,trend
i32,i8,f64,f64
2025,3,761.0,1.206022
2025,4,1419.0,1.864652
2025,8,1169.0,1.562834
2025,9,2075.0,1.775021
2025,10,3980.0,1.918072
2025,12,3192.0,1.479833
2025,16,4825.0,1.204744
2025,22,5643.0,1.223282
2025,24,6543.0,1.456589


## Rychlé přehledy

In [98]:
alt.Chart(
    do_grafu_celorepublika,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('trend:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [99]:
alt.Chart(
    do_grafu_celorepublika_kolo,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('trend:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [100]:
alt.Chart(
    do_grafu_celorepublika_kolo_2w,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('date:T'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [101]:
alt.Chart(
    do_grafu_celorepublika,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [102]:
alt.Chart(
    do_grafu_celorepublika_kolo,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [103]:
do_grafu_celorepublika.drop("trend").write_json("data/graf_rok_behani.json")
do_grafu_celorepublika_kolo.drop("trend").write_json("data/graf_rok_cyklo.json")

## Jednotlivé segmenty

In [105]:
def spocitej_segment(nazev,pojmenovani=""):
    if pojmenovani == "":
        pojmenovani = nazev
    return df.filter(
        (pl.col("year") == 2025) & (pl.col('name') == nazev)
    ).group_by(
        pl.col('date').dt.week()
    ).agg(
        pl.col('effort_diff').sum()
    ).select(
        'date','effort_diff'
    ).with_columns(
        pl.lit(pojmenovani).alias('name')
    ).sort(by='date')

In [106]:
def ukaz_segment(serie):
    return alt.Chart(
        serie
    ).mark_bar(
    ).encode(
        alt.X("date:Q"),
        alt.Y("effort_diff:Q")
    )

In [107]:
ukaz_segment(spocitej_segment("Luční bouda - Sněžka"))

alt.Chart(...)

In [108]:
ukaz_segment(spocitej_segment("Pustevny - Radhošť"))

alt.Chart(...)

In [109]:
ukaz_segment(spocitej_segment("Od posledního tunelu do Bílovic"))

alt.Chart(...)

In [110]:
ukaz_segment(spocitej_segment("Sptint pod Býčí skálou"))

alt.Chart(...)

In [111]:
ukaz_segment(spocitej_segment("Ještěd Finale konec před kostkami."))

alt.Chart(...)

In [112]:
ukaz_segment(spocitej_segment("Revnicak Eve"))

alt.Chart(...)

## Polovina sezony

In [114]:
def serie(segment):

    if isinstance(segment,str):
        segment = [segment]
    
    print(f"Polovina pokusů na segmentu {segment}:")
    
    return df.filter(
        pl.col("date").dt.year() == 2025
    ).filter(
        pl.col("name").is_in(segment)
    ).with_columns(
        pl.col("date").dt.week().alias("tyden")
    ).group_by(
        "tyden"
    ).agg(
        pl.col('effort_diff').sum()
    ).sort(
    by='tyden'
    )

In [115]:
def mark_shortest_half_span(df: pl.DataFrame, value_col: str = "effort_diff", part: int = 2) -> pl.DataFrame:
    values = df[value_col].to_list()
    n = len(values)
    total = sum(values)
    target = total / part

    best_start, best_end = 0, n  # best window (exclusive end)
    window_sum = 0
    left = 0

    for right in range(n):
        window_sum += values[right]
        # Shrink from left as long as we still meet the target
        while window_sum - values[left] >= target:
            window_sum -= values[left]
            left += 1
        # Now check if current window meets target
        if window_sum >= target:
            if (right - left + 1) < (best_end - best_start):
                best_start, best_end = left, right + 1

    half_col = [best_start <= i < best_end for i in range(n)]
    
    return df.with_columns(pl.Series("half", half_col))

In [116]:
def uka(segment):
    return alt.Chart(
    mark_shortest_half_span(df=serie(segment=segment)),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

In [117]:
df.filter(pl.col("country") == "Czechia").group_by(['name','city']).agg(pl.col('effort_diff').median()).sort(by='effort_diff', descending=True)

name,city,effort_diff
str,str,f64
"""Šlechtovka - finální rovinka""","""Praha""",102.0
"""Opičí Časovka, Podolská vodárna - Dvorce""","""Praha""",95.5
"""Die Eine Runde""","""Brno""",92.0
"""Zweite hup u ZOO""","""Praha""",86.0
"""Lužánky rovinka""","""Brno""",84.0
"""Pražačka 320""","""Praha""",70.0
"""Zbraslav - Komořany""","""Zbraslav""",59.5
"""Stadion Na Děkance""","""Praha""",54.0
"""Štvanická lávka (do Karlína)""","""Praha""",48.0


In [118]:
uka(celorepublika)

Polovina pokusů na segmentu ['tenis 1K', 'Smetanovy reverse', 'Hvězda flat', 'Stezka nábřeží', 'stezka', 'Hradní lávka => Sýkorův most', 'mezi mosty', 'Panelovka-Tichá 800m', 'Od Sanusu ke gymplu', 'Ku řopíku', 'Lípy (směrem do Jičína)', 'od hráze až na konec přehrady - silnice', 'Nábřeží', 'Tuhnická lávka -Chebský most', 'U Jordánu do kopce', 'Ciclo Via BNL -> K', 'BK-nabrezi', 'Dlouhá míle', 'Kuželna - Splav 800m', 'po cyklostezce']:


alt.Chart(...)

In [119]:
uka(celorepublika_kolo)

Polovina pokusů na segmentu ['Opičí Časovka, Podolská vodárna - Dvorce', 'Zbraslav - Komořany', 'time trial at Svratka', 'Stezka do Zlina', 'Židlochovice -> Olympia', 'Srbsko - Hlásná Třebáň', 'Revnicak Eve', 'Muglinovská→Hradní lávka (po pravém břehu)', 'Novomlýnská 5: Věstonice-Strachotín', 'svadov—valtirov', 'Jiráskovo nábřeží k Husovce', 'Cyklostezka sprint směr Děčín', 'Nová cyklostezka do Bosche  2', 'Maslovice - Kralupy', 'Adamaov climb', 'Sprintík křižovatka/skalní mlýn', 'Trinec-Tesin', 'Szpic Jested  last 3 km ( od rozc na Dub)', 'Mutěnka - z Kyjova po odbočku na hraběcí', 'Lysá hora - 5 km']:


alt.Chart(...)

In [120]:
uka(celorepublika_kolo_30)

Polovina pokusů na segmentu ['Opičí Časovka, Podolská vodárna - Dvorce', 'Zbraslav - Komořany', 'time trial at Svratka', 'Stezka do Zlina', 'Židlochovice -> Olympia', 'Srbsko - Hlásná Třebáň', 'Revnicak Eve', 'Muglinovská→Hradní lávka (po pravém břehu)', 'Novomlýnská 5: Věstonice-Strachotín', 'svadov—valtirov', 'Jiráskovo nábřeží k Husovce', 'Nová cyklostezka do Bosche  2', 'Cyklostezka sprint směr Děčín', 'Maslovice - Kralupy', 'Adamaov climb', 'Sprintík křižovatka/skalní mlýn', 'Szpic Jested  last 3 km ( od rozc na Dub)', 'Trinec-Tesin', 'Lysá hora - 5 km', 'Mutěnka - z Kyjova po odbočku na hraběcí', 'Údolí Svatý Jan', 'Decin - Bad Schandau Abzweig Radwg', 'Papezov - Prazmo', 'Tuhnice most-Drahovice most,stezka', 'Sedmička, Lipno {', 'Eurovie - nadjezd Rosice', 'Podle Radbuzy', 'Cyklostezka Náchod -> Velké Poříčí', 'MH - Klaster', 'Z Lednice po silnici k odbocce na Janohrad']:


alt.Chart(...)

In [121]:
uka("Lysá hora - 5 km")

Polovina pokusů na segmentu ['Lysá hora - 5 km']:


alt.Chart(...)

In [122]:
uka("Opičí Časovka, Podolská vodárna - Dvorce")

Polovina pokusů na segmentu ['Opičí Časovka, Podolská vodárna - Dvorce']:


alt.Chart(...)

In [123]:
uka("Židlochovice -> Olympia")

Polovina pokusů na segmentu ['Židlochovice -> Olympia']:


alt.Chart(...)

In [124]:
uka("400 m - atletický okruh - Lokomotiva")

Polovina pokusů na segmentu ['400 m - atletický okruh - Lokomotiva']:


alt.Chart(...)

In [187]:
uka(ovaly)

Polovina pokusů na segmentu ['Stadion Na Děkance', 'VUT horní dráha 400m', 'Pražačka 320', 'TJ Sokol okruh', '400m-Liberec', 'Strahovská dráha', 'Tyršák čtyřstofka', '400 m - atletický okruh - Lokomotiva', 'Juliska 400 m', '400m', 'Lužánky - Plochá dráha', 'Ovál ZŠ Štefánikova', '400m - Sportovní areál Na Stoupách', 'Mestsky stadion Trutnov', '400 m (track num. 2)', 'TJ Znojmo Lap', 'Tatran Třemošná', 'Stadion Čelákovice 400m (II.)', '400m-Jičín', 'Hradiště - čtyřstovka']:


alt.Chart(...)

In [175]:
nad1000 = df.filter(
    (pl.col("activity_type") == "Ride") & (pl.col('country') == 'Czechia')
).group_by(
    "name"
).agg(
    pl.col("elevation_high").max(),
    pl.col("effort_diff").mean()
).sort(
    by="elevation_high",descending=True
).filter(
    pl.col('effort_diff') > 2
).filter(
    pl.col('elevation_high') >= 1000
)

print(nad1000)

nad1000 = nad1000.select(pl.col('name')).to_series().to_list()

shape: (10, 3)
┌────────────────────────────────────────────────────┬────────────────┬─────────────┐
│ name                                               ┆ elevation_high ┆ effort_diff │
│ ---                                                ┆ ---            ┆ ---         │
│ str                                                ┆ f64            ┆ f64         │
╞════════════════════════════════════════════════════╪════════════════╪═════════════╡
│ Sedýlko - Praděd climb                             ┆ 1487.0         ┆ 10.349169   │
│ Zlaťák konec - smrt                                ┆ 1383.2         ┆ 4.083532    │
│ Lysá hora - 5 km                                   ┆ 1308.5         ┆ 9.197619    │
│ Výstup na Klínovec                                 ┆ 1241.8         ┆ 6.609639    │
│ Klinovec Top                                       ┆ 1238.8         ┆ 8.596659    │
│ Špindlerovka - část II. (od Černýho potoka nahoru) ┆ 1194.4         ┆ 3.597122    │
│ Do sedla Klínovce po silnici         

In [189]:
uka(nad1000)

Polovina pokusů na segmentu ['Sedýlko - Praděd climb', 'Zlaťák konec - smrt', 'Lysá hora - 5 km', 'Výstup na Klínovec', 'Klinovec Top', 'Špindlerovka - část II. (od Černýho potoka nahoru)', 'Do sedla Klínovce po silnici', 'Kvilda Bučina', 'Kleť k vysílači - finále', 'Bečva - Pustevny Uphill']:


alt.Chart(...)

In [197]:
sezona_nad1000 = mark_shortest_half_span(df=serie(segment=nad1000)).with_columns(pl.lit(f"{len(nad1000)} cyklistických tras nad 1000 m.n.m.").alias("co"))
sezona_ovaly = mark_shortest_half_span(df=serie(segment=ovaly)).with_columns(pl.lit(f"{len(nad1000)} atletických oválů").alias("co"))
sezona_celorepublika = mark_shortest_half_span(df=serie(segment=celorepublika)).with_columns(pl.lit(f"{len(celorepublika)} oblíbených běžeckých úseků").alias("co"))
sezona_celorepublika_kolo = mark_shortest_half_span(df=serie(segment=celorepublika_kolo)).with_columns(pl.lit(f"{len(celorepublika_kolo)} cyklistických úseků").alias("co"))

sezony = pl.concat(
    [
        sezona_nad1000,
        sezona_ovaly,
        sezona_celorepublika,
        sezona_celorepublika_kolo
    ]
)

sezony.write_json('data/tvary_sezon.json')

Polovina pokusů na segmentu ['Sedýlko - Praděd climb', 'Zlaťák konec - smrt', 'Lysá hora - 5 km', 'Výstup na Klínovec', 'Klinovec Top', 'Špindlerovka - část II. (od Černýho potoka nahoru)', 'Do sedla Klínovce po silnici', 'Kvilda Bučina', 'Kleť k vysílači - finále', 'Bečva - Pustevny Uphill']:
Polovina pokusů na segmentu ['Stadion Na Děkance', 'VUT horní dráha 400m', 'Pražačka 320', 'TJ Sokol okruh', '400m-Liberec', 'Strahovská dráha', 'Tyršák čtyřstofka', '400 m - atletický okruh - Lokomotiva', 'Juliska 400 m', '400m', 'Lužánky - Plochá dráha', 'Ovál ZŠ Štefánikova', '400m - Sportovní areál Na Stoupách', 'Mestsky stadion Trutnov', '400 m (track num. 2)', 'TJ Znojmo Lap', 'Tatran Třemošná', 'Stadion Čelákovice 400m (II.)', '400m-Jičín', 'Hradiště - čtyřstovka']:
Polovina pokusů na segmentu ['tenis 1K', 'Smetanovy reverse', 'Hvězda flat', 'Stezka nábřeží', 'stezka', 'Hradní lávka => Sýkorův most', 'mezi mosty', 'Panelovka-Tichá 800m', 'Od Sanusu ke gymplu', 'Ku řopíku', 'Lípy (směrem do